# USD SOFR Swap Monitor

This notebook demonstrates how to use SDRUtils to:
1. Fetch and filter USD SOFR swap trades from DTCC SDR
2. Classify trades with tenor, forward start, and PV01
3. Detect multi-leg packages (FLY, CURVE, SPREADOVER)
4. Analyze and visualize trade flows

## Setup

In [1]:
%load_ext autoreload
%autoreload 2

import nest_asyncio
nest_asyncio.apply()

# Plotting
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "notebook"

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
plt.style.use('seaborn-v0_8-dark')
pylab.rcParams.update({
    'legend.fontsize': 'x-large',
    'figure.figsize': (16, 9),
    'axes.labelsize': 'x-large',
    'axes.titlesize': 'x-large',
    'xtick.labelsize': 'x-large',
    'ytick.labelsize': 'x-large'
})

# Core
import pandas as pd
import numpy as np
import datetime
import pytz
from tqdm import tqdm

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

# Timezones
NY_tz = pytz.timezone("America/New_York")
UTC_tz = pytz.timezone("UTC")

## Import SDRUtils Components

In [ ]:
# Data fetching
from SDRUtils.data.builder import SDRDataBuilder

# USD SOFR swap classification
from SDRUtils.products.usd import classify_sofr_swap_trade
from SDRUtils.products.usd.filters import new_sofr_swap_trades

# Core classification utilities
from SDRUtils.core.classification import classifications_to_dataframe

# Package detection
from SDRUtils.packages import (
    detect_fly_trades_df,
    detect_curve_trades_df,
    detect_spreadover_trades_df,
    merge_package_legs_to_one_row,
)

# Analytics
from SDRUtils.analytics import (
    add_event_classifications,
    aggregate_flows_by_label,
)

# Curve for PV01 calculation
from MDP.IRSwaps.IRSwapsMDP import IRSwapsMDP

print("SDRUtils components loaded successfully!")

## Configuration

In [ ]:
# Cache path for SDR data
CACHE_PATH = r"C:\Users\chris\clee\project-oasis\private\sdranalytics\.cache"

# Time range for analysis
START_TIME = NY_tz.localize(datetime.datetime(2025, 12, 29, 0, 1))
END_TIME = NY_tz.localize(datetime.datetime(2025, 12, 29, 20, 0))

print(f"Analysis period: {START_TIME} to {END_TIME}")

## 1. Fetch Raw SDR Data

In [ ]:
# Initialize SDR data builder
sdr = SDRDataBuilder(cache_path=CACHE_PATH, show_tqdm=True)

# Fetch and filter to SOFR swaps
raw_df = sdr.grab_sdr_trades(
    start_timestamp=START_TIME,
    end_timestamp=END_TIME,
    agency="CFTC",
    asset_class="RATES",
    filter_func=new_sofr_swap_trades,
)

print(f"Fetched {len(raw_df):,} SOFR swap trades")
raw_df.head()

## 2. Initialize Curve for PV01 Calculation

In [ ]:
# Get SOFR curve for PV01 calculation
swaps_mdp = IRSwapsMDP(source="ERIS_EOD_LIVE-RL_BASIC")
curve = swaps_mdp.get_pricer(
    request=dict(
        curve_name="USD-SOFR-1D",
        timestamp=START_TIME.date()
    )
)

print(f"Curve initialized for {START_TIME.date()}")

## 3. Classify All Trades

In [ ]:
# Classify each trade
classifications = []
failed_trades = []

for idx, row in tqdm(raw_df.iterrows(), total=len(raw_df), desc="Classifying trades"):
    trade_id = int(row.get("Dissemination Identifier", idx))
    try:
        classification = classify_sofr_swap_trade(row, trade_id, curve)
        classifications.append(classification)
    except Exception as e:
        failed_trades.append((trade_id, str(e)))
        continue

# Convert to DataFrame
classified_df = classifications_to_dataframe(classifications)

print(f"Successfully classified {len(classified_df):,} trades")
print(f"Failed to classify {len(failed_trades)} trades")
classified_df.head()

## 4. Detect All Package Types

In [ ]:
# Apply package detection in sequence:
# 1. FLY detection (3-leg butterflies)
# 2. CURVE detection (2-leg curve trades)
# 3. SPREADOVER detection (swap/UST matched maturity)

print("Detecting FLY packages...")
with_fly_df = detect_fly_trades_df(
    classified_df,
    time_window_seconds=60,
    belly_ratio_tolerance=0.15,
)

print("Detecting CURVE packages...")
with_curve_df = detect_curve_trades_df(
    with_fly_df,
    time_window_seconds=60,
    pv01_tolerance=0.10,
)

print("Detecting SPREADOVER packages...")
with_all_pkg_df = detect_spreadover_trades_df(
    with_curve_df,
    only_tag_outrights=True,
)

# Summary of package types
print("\nPackage type distribution:")
print(with_all_pkg_df['package_type'].value_counts())

## 5. Merge Multi-Leg Packages into Single Rows

In [ ]:
# Collapse multi-leg packages (FLY, CURVE) into single rows
merged_df = merge_package_legs_to_one_row(with_all_pkg_df)

print(f"Trades after merging packages: {len(merged_df):,} (from {len(with_all_pkg_df):,})")
print("\nPackage type distribution (after merge):")
print(merged_df['package_type'].value_counts())

## 6. Analysis by Package Type

In [ ]:
# Analyze FLY trades
fly_trades = merged_df[merged_df['package_type'] == 'FLY'].copy()
print(f"\n=== FLY Trades ({len(fly_trades)}) ===")
if len(fly_trades) > 0:
    display(fly_trades[['trade_label', 'notional', 'estimated_pv01', 'fixed_rate', 'package_id']].head(10))

In [ ]:
# Analyze CURVE trades
curve_trades = merged_df[merged_df['package_type'] == 'CURVE'].copy()
print(f"\n=== CURVE Trades ({len(curve_trades)}) ===")
if len(curve_trades) > 0:
    display(curve_trades[['trade_label', 'notional', 'estimated_pv01', 'fixed_rate', 'package_id']].head(10))

In [ ]:
# Analyze SPREADOVER trades (matched UST maturity)
spreadover_trades = merged_df[merged_df['package_type'] == 'SPREADOVER'].copy()
print(f"\n=== SPREADOVER Trades ({len(spreadover_trades)}) ===")
if len(spreadover_trades) > 0:
    display(spreadover_trades[[
        'trade_label', 'notional', 'estimated_pv01', 'fixed_rate',
        'ust_cusip', 'ust_oi', 'swap_maturity_date'
    ]].head(10))

In [ ]:
# Analyze OUTRIGHT trades
outright_trades = merged_df[merged_df['package_type'] == 'OUTRIGHT'].copy()
print(f"\n=== OUTRIGHT Trades ({len(outright_trades)}) ===")
if len(outright_trades) > 0:
    display(outright_trades[['trade_label', 'notional', 'estimated_pv01', 'fixed_rate']].head(10))

## 7. Visualizations

In [ ]:
# PV01 by Package Type
fig = px.box(
    merged_df,
    x='package_type',
    y='estimated_pv01',
    title='PV01 Distribution by Package Type',
    labels={'estimated_pv01': 'Estimated PV01 ($)', 'package_type': 'Package Type'},
)
fig.update_layout(height=500)
fig.show()

In [ ]:
# Volume by Tenor Label
# For merged packages, split the trade_label to get individual tenors
tenor_volume = merged_df.groupby('tenor_label').agg({
    'notional': 'sum',
    'estimated_pv01': 'sum',
    'trade_id': 'count'
}).rename(columns={'trade_id': 'trade_count'}).sort_values('notional', ascending=False)

fig = px.bar(
    tenor_volume.reset_index().head(15),
    x='tenor_label',
    y='notional',
    title='Notional Volume by Tenor',
    labels={'notional': 'Total Notional ($)', 'tenor_label': 'Tenor'},
)
fig.update_layout(height=500)
fig.show()

In [ ]:
# Timeline of trades by package type
merged_df['exec_hour'] = pd.to_datetime(merged_df['execution_timestamp']).dt.hour

hourly_counts = merged_df.groupby(['exec_hour', 'package_type']).size().unstack(fill_value=0)

fig = px.bar(
    hourly_counts.reset_index().melt(id_vars='exec_hour'),
    x='exec_hour',
    y='value',
    color='package_type',
    title='Trade Count by Hour and Package Type',
    labels={'value': 'Trade Count', 'exec_hour': 'Hour (UTC)', 'package_type': 'Package Type'},
    barmode='stack',
)
fig.update_layout(height=500)
fig.show()

In [ ]:
# Fixed Rate Distribution by Tenor (for spot trades)
spot_trades = merged_df[
    (merged_df['forward_label'] == 'spot') & 
    (merged_df['package_type'] == 'OUTRIGHT')
].copy()

# Filter to standard tenors
standard_tenors = ['2Y', '3Y', '5Y', '7Y', '10Y', '20Y', '30Y']
spot_trades = spot_trades[spot_trades['tenor_label'].isin(standard_tenors)]

if len(spot_trades) > 0:
    fig = px.box(
        spot_trades,
        x='tenor_label',
        y='fixed_rate',
        title='Fixed Rate Distribution by Tenor (Spot Outrights)',
        labels={'fixed_rate': 'Fixed Rate', 'tenor_label': 'Tenor'},
        category_orders={'tenor_label': standard_tenors},
    )
    fig.update_layout(height=500)
    fig.show()

## 8. SPREADOVER Analysis (Swap vs UST)

In [ ]:
# Detailed SPREADOVER analysis
if len(spreadover_trades) > 0:
    # Group by UST original issue
    ust_summary = spreadover_trades.groupby('ust_oi').agg({
        'notional': 'sum',
        'estimated_pv01': 'sum',
        'trade_id': 'count',
        'fixed_rate': 'mean',
    }).rename(columns={'trade_id': 'trade_count', 'fixed_rate': 'avg_rate'})
    
    ust_summary = ust_summary.sort_values('notional', ascending=False)
    
    print("SPREADOVER Summary by UST Original Issue:")
    display(ust_summary)
    
    # Bar chart
    fig = px.bar(
        ust_summary.reset_index(),
        x='ust_oi',
        y='notional',
        title='SPREADOVER Volume by UST Original Issue',
        labels={'notional': 'Total Notional ($)', 'ust_oi': 'UST Original Issue'},
    )
    fig.update_layout(height=400)
    fig.show()
else:
    print("No SPREADOVER trades found")

## 9. Summary Statistics

In [ ]:
# Overall summary
print("=" * 60)
print("USD SOFR SWAP MONITOR - SUMMARY")
print("=" * 60)
print(f"\nPeriod: {START_TIME} to {END_TIME}")
print(f"\nTotal trades fetched: {len(raw_df):,}")
print(f"Successfully classified: {len(classified_df):,}")
print(f"After package merge: {len(merged_df):,}")

print("\n" + "-" * 40)
print("PACKAGE BREAKDOWN")
print("-" * 40)
pkg_summary = merged_df.groupby('package_type').agg({
    'notional': ['sum', 'mean'],
    'estimated_pv01': ['sum', 'mean'],
    'trade_id': 'count'
})
pkg_summary.columns = ['Total Notional', 'Avg Notional', 'Total PV01', 'Avg PV01', 'Count']
display(pkg_summary)

print("\n" + "-" * 40)
print("TOP 10 TRADE LABELS BY VOLUME")
print("-" * 40)
top_labels = aggregate_flows_by_label(merged_df, value_col='notional').head(10)
display(top_labels.to_frame('Total Notional'))

print("\n" + "=" * 60)

## 10. Export Results

In [ ]:
# Export to CSV if needed
# merged_df.to_csv('sofr_swap_monitor_results.csv', index=False)
# print("Results exported to sofr_swap_monitor_results.csv")

# Or display final DataFrame
print("Final merged DataFrame columns:")
print(merged_df.columns.tolist())

In [ ]:
# Sample of final output
display_cols = [
    'trade_id', 'execution_timestamp', 'trade_label', 'product_type',
    'notional', 'fixed_rate', 'estimated_pv01',
    'package_type', 'package_id',
    'matched_ust_maturity', 'ust_oi'
]
available_cols = [c for c in display_cols if c in merged_df.columns]
merged_df[available_cols].head(20)